In [1]:
import numpy as np
import pandas as pd
from numpy.char import lower

INPUT_FILE = "D:\\Work\\Climate Adaption Planner\\Datasets\\Crop_recommendationV1.csv"
INPUT_FILE2 = "D:\\Work\\Climate Adaption Planner\\Datasets\\Crop_recommendationV2.csv"
OUTPUT_FILE = "D:\\Work\\Climate Adaption Planner\\SmartFarming\\Data_Setup\\Datasets\\crop_recommendationV3.csv" 

crops_needed = ['rice','maize','cotton', 'wheat']

In [2]:
def read_crop_data(file_path):
    """
    Reads crop data from a CSV file and returns it as a pandas DataFrame.

    Parameters:
    file_path (str): The path to the CSV file containing crop data.

    Returns:
    pd.DataFrame: A DataFrame containing the crop data.
    """
    try:
        crop_data = pd.read_csv(file_path)
        return crop_data
    except Exception as e:
        print(f"An error occurred while reading the crop data: {e}")
        return None

In [3]:
crop_data = read_crop_data(INPUT_FILE)

columns = crop_data.columns
print("Columns in the crop data:", columns)

related_data = crop_data[crop_data['label'].str.lower().isin(crops_needed)]
print("Data for requested crops:")
print(related_data.groupby('label').size())

unique_crops = related_data['label'].unique()
print("Unique crops in the dataset:", unique_crops.tolist())

unique_weather = crop_data['season'].unique()
print("Unique seasons in the dataset:", unique_weather.tolist())


Columns in the crop data: Index(['temperature', 'humidity', 'ph', 'water availability', 'season',
       'label'],
      dtype='str')
Data for requested crops:
label
cotton    100
maize     200
rice      100
dtype: int64
Unique crops in the dataset: ['rice', 'maize', 'cotton']
Unique seasons in the dataset: ['rainy', 'winter', 'spring', 'summer']


In [4]:
crop_data2 = read_crop_data(INPUT_FILE2)

columns = crop_data2.columns
print("Columns in the crop data:", columns)

related_data2 = crop_data2[crop_data2['label'].str.lower().isin(crops_needed)]
print("Data for requested crops:")
print(related_data2.groupby('label').size())

unique_crops = related_data2['label'].unique()
print("Unique crops in the dataset:", unique_crops.tolist())

unique_weather = crop_data2['growth_stage'].unique()
print("Unique growth stages in the dataset:", unique_weather.tolist())

Columns in the crop data: Index(['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label',
       'soil_moisture', 'soil_type', 'sunlight_exposure', 'wind_speed',
       'co2_concentration', 'organic_matter', 'irrigation_frequency',
       'crop_density', 'pest_pressure', 'fertilizer_usage', 'growth_stage',
       'urban_area_proximity', 'water_source_type', 'frost_risk',
       'water_usage_efficiency'],
      dtype='str')
Data for requested crops:
label
cotton    100
maize     100
rice      100
dtype: int64
Unique crops in the dataset: ['rice', 'maize', 'cotton']
Unique growth stages in the dataset: [1, 3, 2]


In [5]:
# total records in both datasets
crop1_records = len(crop_data)
crop2_records = len(crop_data2)
total_records = crop2_records - crop1_records
print(f"Total records diff in both datasets: {total_records}, Crop1 records: {crop1_records}, Crop2 records: {crop2_records}")

# per crop records in both datasets
crop1_counts = crop_data['label'].str.lower().value_counts()
crop2_counts = crop_data2['label'].str.lower().value_counts()

# unique crops in both datasets
unique_crops1 = set(crop_data['label'].str.lower().unique())
unique_crops2 = set(crop_data2['label'].str.lower().unique())

print("Crop counts in dataset 1:", crop1_counts.count())
print("Crop counts in dataset 2:", crop2_counts.count())
print(f"Difference in crop_counts: {crop2_counts.count() - crop1_counts.count()}")
print("\n")
print(f"Crops in both datasets: {sorted(unique_crops1.intersection(unique_crops2))}")
print(f"Crops only in dataset 1: {sorted(unique_crops1.difference(unique_crops2))}")
print(f"Crops only in dataset 2: {sorted(unique_crops2.difference(unique_crops1))}")
print("\n")

# Difference in crop count for same crops in both datasets
for crop in sorted(unique_crops1.intersection(unique_crops2)):
    count1 = crop1_counts.get(crop, 0)
    count2 = crop2_counts.get(crop, 0)
    print(f"Crop: {crop}, Difference: {count2 - count1}")

# Common columns in both datasets
common_columns = set(crop_data.columns).intersection(set(crop_data2.columns))  
print(f"Common columns in both datasets: {sorted(common_columns)}")


Total records diff in both datasets: 800, Crop1 records: 1400, Crop2 records: 2200
Crop counts in dataset 1: 13
Crop counts in dataset 2: 22
Difference in crop_counts: 9


Crops in both datasets: ['blackgram', 'chickpea', 'cotton', 'jute', 'kidneybeans', 'lentil', 'maize', 'mothbeans', 'mungbean', 'muskmelon', 'pigeonpeas', 'rice', 'watermelon']
Crops only in dataset 1: []
Crops only in dataset 2: ['apple', 'banana', 'coconut', 'coffee', 'grapes', 'mango', 'orange', 'papaya', 'pomegranate']


Crop: blackgram, Difference: 0
Crop: chickpea, Difference: 0
Crop: cotton, Difference: 0
Crop: jute, Difference: 0
Crop: kidneybeans, Difference: 0
Crop: lentil, Difference: 0
Crop: maize, Difference: -100
Crop: mothbeans, Difference: 0
Crop: mungbean, Difference: 0
Crop: muskmelon, Difference: 0
Crop: pigeonpeas, Difference: 0
Crop: rice, Difference: 0
Crop: watermelon, Difference: 0
Common columns in both datasets: ['humidity', 'label', 'ph', 'temperature']


In [6]:
# columns where water_availability in dataset 1 = rainfall in dataset 2
# condition => crop1.label = crop2.label and crop1.humidity = crop2.humidity and crop1.temperature = crop2.temperature and crop1.ph = crop2.ph
# keep all data from related_data2 and match with related_data based on the above condition
cols = ['label', 'humidity', 'temperature', 'ph', 'season']

d_grouped = (
    related_data[cols]
    .groupby(['label', 'humidity', 'temperature', 'ph'])['season']
    .agg(lambda x: ', '.join(sorted(set(x))))
    .reset_index()
)

merged_data = pd.merge(
    related_data2,
    d_grouped,
    on=['label', 'humidity', 'temperature', 'ph'],
    how='left'
)
merged_data.to_csv(OUTPUT_FILE,  index=False)

In [8]:
merged_data.drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)
print("Per Crop Rows in Merged Data")
print(merged_data.groupby('label').size())
print(f"Number of nulls in Season: {merged_data['season'].isnull().sum()}")
print("Columns in merged data:", merged_data.columns)


Per Crop Rows in Merged Data
label
cotton    100
maize     100
rice      100
dtype: int64
Number of nulls in Season: 0
Columns in merged data: Index(['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label',
       'soil_moisture', 'soil_type', 'sunlight_exposure', 'wind_speed',
       'co2_concentration', 'organic_matter', 'irrigation_frequency',
       'crop_density', 'pest_pressure', 'fertilizer_usage', 'growth_stage',
       'urban_area_proximity', 'water_source_type', 'frost_risk',
       'water_usage_efficiency', 'season'],
      dtype='str')


In [ ]:
"""Get unique values for Master Vocabulary Tables
1. Crops
2. Growth_Stage
3. Soil_Type
4. Season
5. Water Sources
"""

